# Résultats des simulations : comparaison aux données empiriques

**Question.** Le modèle de transmission culturelle avec archive cumulative (innovation `mu`, biais de conformité `q`, distribution initiale en loi de puissance d'exposant `alpha` sur `C` classes) peut-il reproduire la distribution des catégories (tags) du corpus fandom, et pour quels paramètres ?

**Données empiriques** (`data/fandom_data.csv`) : 13 128 histoires, 176 catégories, 30 901 occurrences de catégories (une histoire porte en moyenne 2,35 catégories).

**Simulations** (`run_grid.py`) : deux grilles de 5 000 000 de runs, paramètres tirés uniformément et indépendamment.

| Grille | `n_f` (taille finale de l'archive) | Paramètres tirés |
|---|---|---|
| `n_f = 10 000` | 10 000 | `q` ∈ [0,5 ; 1,5], `mu` ∈ [0 ; 0,05], `alpha` ∈ [0 ; 2], `C` ∈ [2 ; 200], `n_i` ∈ [1 ; 100], `T` ∈ [300 ; 1000] |
| `n_f = 30 901` | 30 901 (= occurrences empiriques) | idem, sauf `mu` ∈ [0 ; 0,02] (rééchelonné : `mu × n_f` ≈ nombre de classes créées) |

## Choix de métriques

Toutes sont calculées sur l'archive **finale** (comptes cumulés par classe, `n_f` individus), sans identité de classe commune avec les données : on ne compare que des propriétés de la **distribution des tailles de classes**.

| Indice | Ce qu'il mesure | Pourquoi ce choix |
|---|---|---|
| `gini_obs` | inégalité de la répartition, **classes observées seulement** (compte > 0) | Comparable à l'empirique, où les 176 catégories sont toutes observées. C'est l'indice à utiliser. |
| `gini` | idem, mais compte aussi les classes initiales jamais échantillonnées (compte 0) | Conservé pour mémoire : il est gonflé par ces classes fantômes (présentes dans 96 % des runs) et a biaisé une première sélection. |
| Hill d'ordre 0, 1, 2, ∞ (`hill_0`…`hill_inf`) | nombre « effectif » de classes, de plus en plus centré sur les classes dominantes | Un même profil complet (richesse → dominance) plutôt qu'un seul nombre. `hill_0` = richesse observée. |
| `chao1` | richesse totale estimée (classes non vues, d'après singletons et doubletons) | Sensible aux classes rares, que Hill 1 à ∞ ignorent. |
| `kl_emp` | divergence KL(empirique ‖ simulation) entre distributions **rang-fréquence** (176 rangs, plancher de 0,5 occurrence pour les rangs non atteints) | Compare la **forme** complète de la distribution, pas seulement des résumés. Fréquences relatives : indépendant de la taille d'échantillon. Bruit d'échantillonnage : KL(empirique ‖ empirique aminci) ≈ 0,001. |
| Exposant MLE de loi de puissance (`alpha_hat`, `xmin`, `n_tail`) | exposant d'une loi de puissance discrète ajustée sur la queue (Clauset-Shalizi-Newman, `xmin` par minimisation KS) | Estimateur de `analysis.fit_power_law_mle` (le même que `plot_archive_rate_mle.py` de l'ancien modèle) : test spécifique de la forme en loi de puissance de l'empirique (`alpha_hat` = 1,53 ± 0,05). Dépend de la taille d'échantillon (d'où l'empirique aminci à 10 000). |

**Sélection des « meilleurs runs ».** Deux critères complémentaires : (1) **ABC** : distance euclidienne en unités d'écart-type sur `gini_obs`, `hill_1`, `hill_2`, `hill_inf` et `log(chao1)` (log car queue très lourde) ; (2) **KL minimal** : forme de la distribution. Comparer les deux montre si des indices scalaires ajustés suffisent à reproduire la forme.

## Choix de visualisation
- **Pair plot** (`Explorer.pairplot`) : voir d'un coup toutes les relations paramètres ↔ indices et indices ↔ indices. Les points sont colorés par la distance à l'empirique quand elle existe ; les lignes rouges marquent les valeurs empiriques. Hill, Chao1 et `kl_emp` sont tracés en **log10** (queues très lourdes). Un échantillon de 3 000 runs suffit à lire les structures.
- **`Explorer.index`** : examen détaillé d'un indice : distribution et médiane avec bande 10-90 % par tranche de chaque paramètre, pour lire l'effet marginal d'un paramètre.
- **Rang-fréquence en log-log** avec droites MLE : forme de la distribution, comparable directement à l'empirique. Comptes bruts quand les tailles d'échantillon sont égales (`n_f = 30 901`), fréquences relatives sinon.
- **Tableaux de quantiles** (5 %, médiane, 95 %) plutôt que des moyennes : les distributions sont asymétriques.

## Comment lire ce notebook
Exécuter dans l'ordre. Les blocs de simulation (`.npz`) sont lus dans `../results/grid_n10000/chunks/` et `../results/grid_n30901/chunks/` (rapatriés du serveur : `rsync -a --include='chunk_*.npz' --include='grid_config.json' --exclude='*' les-lillas:<grille>/ results/<dossier>/chunks/`, cf. README). Les paramètres exacts (`alpha`, `mu`, `q`) sont lus en float64 : la simulation est très sensible au dernier chiffre, ce qui permet de rejouer un run. Les conclusions sont dans la cellule finale.

## 0. Préparation : chargement des deux grilles

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.special import zeta as hurwitz_zeta

sys.path.insert(0, str(Path("..").resolve()))      # racine du repo : simulation, run_grid, analysis
from run_grid import load_results, load_counts
from simulation import Simulation, compute_metrics
from analysis import Explorer

CHUNK_SIZE = 5000                                   # --chunk_size utilisé pour les grilles
N_SIMS = 5_000_000
GRIDS = {                                            # dossier des blocs, n_f, borne haute de mu tirée
    10_000: (Path("../results/grid_n10000/chunks"), 10_000, 0.05),
    30_901: (Path("../results/grid_n30901/chunks"), 30_901, 0.02),
}


def load_grid(grid_dir, n_f, mu_max):
    '''Charge une grille (blocs .npz) en DataFrame, avec contrôles ; paramètres alpha, mu, q exacts en float64.'''
    n_chunks = len(list(grid_dir.glob("chunk_*.npz")))
    assert n_chunks == N_SIMS // CHUNK_SIZE, f"{n_chunks} blocs dans {grid_dir.resolve()} : rapatrier les blocs manquants"
    df = pd.DataFrame(load_results(grid_dir)).astype({
        "C": "int16", "n_i": "int16", "n_f": "int32", "T": "int16", "seed": "int32", "n_classes": "int16",
        "gini": "float32", "gini_obs": "float32", "hill_0": "float32", "hill_1": "float32", "hill_2": "float32",
        "hill_inf": "float32", "chao1": "float32", "kl_emp": "float32",
    })
    assert len(df) == N_SIMS and (df["seed"].to_numpy() == np.arange(1, N_SIMS + 1)).all()   # ordre des runs : seed = ligne + 1
    assert (df["n_f"] == n_f).all() and df["mu"].max() > 0.99 * mu_max, f"ce n'est pas la grille n_f = {n_f}"
    return df


df10, df30 = (load_grid(*GRIDS[k]) for k in (10_000, 30_901))
ex10, ex30 = Explorer(df10), Explorer(df30)
print(ex10); print(ex30)

## 1. Indices empiriques

Calculés sur `data/fandom_data.csv` (catégories combinées `A|B` éclatées), avec les mêmes fonctions que les simulations. Chao1 = S_obs + f1² / (2·f2) = 176 + 12² / (2·3) = **200** (12 catégories à une occurrence, 3 à deux). Le Gini empirique est le même pour `gini` et `gini_obs` (toutes les catégories sont observées).

In [ ]:
# Valeurs écrites en dur (calculées une fois sur data/fandom_data.csv)
empirical = {
    "gini": 0.7963273842270476, "gini_obs": 0.7963273842270476,
    "hill_0": 176.0, "hill_1": 45.506516340063556, "hill_2": 26.53617935151933, "hill_inf": 9.784990500316656,
    "chao1": 200.0, "n_classes": 176, "kl_emp": 0.0,
}
empirical_info = {"n_stories": 13128, "n_occurrences": 30901, "f1_singletons": 12, "f2_doubletons": 3, "count_max": 3158}

# Comptes empiriques par catégorie (catégories combinées éclatées), triés décroissant
emp_counts = np.array([
    3158, 2698, 1688, 1269, 1263, 1244, 1198, 1090, 1051, 1043, 975, 932, 764, 724, 621, 587, 557, 542, 540, 488,
    462, 401, 313, 309, 290, 281, 272, 264, 243, 237, 216, 199, 190, 181, 178, 171, 161, 155, 144, 141,
    134, 133, 131, 114, 113, 112, 108, 100, 100, 98, 96, 86, 79, 69, 68, 66, 58, 53, 52, 49,
    48, 47, 46, 45, 43, 41, 40, 36, 35, 34, 33, 33, 33, 33, 32, 31, 29, 29, 29, 28,
    28, 28, 28, 26, 26, 25, 24, 24, 23, 23, 23, 21, 21, 20, 20, 20, 20, 20, 19, 19,
    18, 18, 17, 17, 17, 16, 16, 16, 16, 16, 15, 15, 15, 15, 15, 15, 15, 15, 14, 14,
    13, 13, 13, 12, 12, 12, 12, 11, 11, 11, 11, 11, 11, 11, 10, 10, 10, 9, 9, 8,
    7, 7, 7, 7, 7, 6, 6, 6, 6, 6, 6, 5, 4, 4, 4, 3, 3, 3, 3, 3,
    3, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
])
assert len(emp_counts) == 176 and emp_counts.sum() == 30901

## 2. Outils : ajustement MLE, sélection des meilleurs runs, figures

L'ajustement MLE de loi de puissance (Clauset, Shalizi & Newman 2009, `xmin` par minimisation KS) est dans `analysis.py`.

In [ ]:
# ------------------------------------------------ MLE loi de puissance discrète (Clauset, Shalizi & Newman 2009)
from analysis import fit_power_law_mle, draw_fit

emp_fit = fit_power_law_mle(emp_counts)
print("empirique :", {k: round(float(v), 3) for k, v in emp_fit.items()})

def qtl(x):
    x = pd.Series(x); return f"{x.median():.2f} [{x.quantile(.05):.2f} ; {x.quantile(.95):.2f}]"

# ------------------------------------------------ sélection des meilleurs runs
METRICS_ABC = ("gini_obs", "hill_1", "hill_2", "hill_inf", "chao1")     # chao1 passé en log dans Explorer.distance
SEL_COLORS = {"ABC (gini_obs, Hill, Chao1)": "tab:blue", "KL minimal": "tab:green"}

def select_top(ex, df, n_top=100):
    '''Deux sélections des n_top meilleurs runs : distance ABC (indices scalaires) et KL minimal (forme).'''
    return {"ABC (gini_obs, Hill, Chao1)": ex.closest(n_top, metrics=METRICS_ABC).df.sort_values("distance"),
            "KL minimal": df.nsmallest(n_top, "kl_emp")}

def read_counts(sel, grid_dir, n_f):
    '''Distributions complètes (comptes triés décroissant) de chaque run sélectionné, avec contrôle de cohérence.'''
    out = {}
    for name, d in sel.items():
        cs = [load_counts(grid_dir, int(s) - 1, CHUNK_SIZE) for s in d["seed"]]
        for c, (_, r) in zip(cs, d.iterrows()):
            m = compute_metrics(c)
            assert c.sum() == n_f
            assert all(abs(m[k] - r[k]) <= 1e-4 * max(abs(m[k]), 1e-9) for k in ["gini_obs", "hill_1", "hill_2", "hill_inf", "chao1", "kl_emp"]), (name, r["seed"])
        out[name] = cs
    return out

def fit_all(counts):
    return {name: pd.DataFrame([f for f in (fit_power_law_mle(c) for c in cs) if f is not None]) for name, cs in counts.items()}

def thinned_empirical_fits(n, reps=300, seed=0):
    '''Ajustements de l'empirique aminci à n occurrences (tirage sans remise), pour comparer à taille égale.'''
    rng = np.random.default_rng(seed)
    return pd.DataFrame([fit_power_law_mle(np.sort(rng.multivariate_hypergeometric(emp_counts, n))[::-1]) for _ in range(reps)])

def report_fits(fits, lo, hi, ref_label):
    '''Tableau des exposants MLE des sélections ; [lo, hi] = intervalle de référence de l'exposant empirique.'''
    print(f"{'sélection':30s}{'alpha':>22s}{'xmin':>22s}{'n_tail':>24s}   alpha dans {ref_label}")
    for name, f in fits.items():
        print(f"{name:30s}{qtl(f.alpha):>22s}{qtl(f.xmin):>22s}{qtl(f.n_tail):>24s}   {((f.alpha >= lo) & (f.alpha <= hi)).sum()} / {len(f)}")

def selection_summary(sel, n_f):
    cols = ["q", "mu", "alpha", "C", "n_i", "T", "gini_obs", "hill_0", "hill_1", "hill_2", "hill_inf", "chao1", "kl_emp"]
    t = pd.DataFrame({name: d[cols].median() for name, d in sel.items()}).T
    t.insert(2, "mu x n_f", t["mu"] * n_f)
    t.loc["empirique", ["gini_obs", "hill_0", "hill_1", "hill_2", "hill_inf", "chao1", "kl_emp"]] = [
        empirical[k] for k in ["gini_obs", "hill_0", "hill_1", "hill_2", "hill_inf", "chao1", "kl_emp"]]
    return t.round(4)

def plot_rank_frequency(counts, fits, n_f, thin=None):
    '''Rang-fréquence des sélections vs empirique (droites MLE) + histogramme des exposants. Comptes bruts si n_f égale la taille
    de l'échantillon empirique, sinon fréquences relatives.'''
    relative = n_f != emp_counts.sum()
    s_sim, s_emp = (n_f, emp_counts.sum()) if relative else (1, 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    ranks_e = np.arange(1, len(emp_counts) + 1)
    for ax, (name, cs) in zip(axes[:2], counts.items()):
        for c in cs:
            cp = c[c > 0]; ax.loglog(np.arange(1, len(cp) + 1), cp / s_sim, color=SEL_COLORS[name], lw=0.6, alpha=0.35)
        padded = np.zeros((len(cs), max(len(c) for c in cs)))
        for i, c in enumerate(cs):
            padded[i, :len(c)] = c
        med = np.round(np.median(padded, axis=0)); med = med[med > 0]; med_fit = fit_power_law_mle(med)
        ax.loglog(np.arange(1, len(med) + 1), med / s_sim, color="tab:red", lw=2.5, label="médiane des runs")
        draw_fit(ax, med, med_fit, s_sim, color="tab:red", lw=2, ls="--", label=fr"MLE médiane ($\hat\alpha$={med_fit['alpha']:.2f}$\pm${med_fit['sigma']:.2f})")
        ax.loglog(ranks_e, emp_counts / s_emp, "D", ms=5, color="black", mfc="none", mew=1.3, label="empirique")
        draw_fit(ax, emp_counts, emp_fit, s_emp, color="black", lw=1.8, ls="--", label=fr"MLE empirique ($\hat\alpha$={emp_fit['alpha']:.2f}$\pm${emp_fit['sigma']:.2f})")
        ax.set_xlabel("rang"); ax.set_ylabel("fréquence relative" if relative else f"comptes bruts (n = {n_f:,})")
        ax.set_title(f"{name}\n{len(cs)} meilleurs runs"); ax.grid(True, which="both", ls=":", lw=0.5, alpha=0.4); ax.legend(fontsize=8)
    ax = axes[2]
    bins = np.histogram_bin_edges(np.concatenate([f.alpha.clip(upper=4) for f in fits.values()]), bins=25)
    if thin is not None:
        ax.hist(thin.alpha.clip(upper=4), bins=bins, color="black", alpha=0.5, label="empirique aminci à n_f")
    for name, f in fits.items():
        ax.hist(f.alpha.clip(upper=4), bins=bins, alpha=0.6, color=SEL_COLORS[name], label=name)
    ax.axvline(emp_fit["alpha"], color="black", lw=2, label=fr"empirique {emp_fit['alpha']:.2f} ± {emp_fit['sigma']:.2f}")
    ax.set_xlabel(r"exposant MLE $\hat\alpha$ (tronqué à 4)"); ax.set_ylabel("nombre de runs"); ax.set_title("Exposants ajustés"); ax.legend(fontsize=8)
    fig.tight_layout(); plt.show()

def identifiability(ex, df, ranges, n_top=1000):
    '''Quantiles des paramètres parmi les n_top meilleurs runs de chaque sélection, par rapport à la plage tirée.'''
    rows = []
    for name, d in {"ABC (gini_obs, Hill, Chao1)": ex.closest(n_top, metrics=METRICS_ABC).df, "KL minimal": df.nsmallest(n_top, "kl_emp")}.items():
        for p, (lo, hi) in ranges.items():
            qs = d[p].quantile([0.05, 0.5, 0.95]).to_numpy()
            rows.append({"sélection": name, "paramètre": p, "5 %": qs[0], "médiane": qs[1], "95 %": qs[2], "largeur 5-95 % / plage tirée": (qs[2] - qs[0]) / (hi - lo)})
    return pd.DataFrame(rows).set_index(["sélection", "paramètre"]).round(4)

## 3. Vérification des résultats

Trois contrôles sur chaque grille : propriétés qui doivent toujours être vraies, uniformité des paramètres tirés, et rejeu de runs à partir de leurs paramètres exacts et de leur graine. Le rejeu local (numpy 2.x) peut différer du serveur (numpy 1.26) sur environ 1 % des runs : pow/binomial/multinomial ne sont pas garantis identiques d'une version à l'autre et la simulation amplifie la moindre différence. Les données stockées restent cohérentes entre elles.

In [ ]:
def check_properties(df, name):
    tol = 1 - 1e-5   # marge float32
    checks = {
        "aucune valeur manquante": int(df.isna().sum().sum()) == 0,
        "gini et gini_obs dans [0, 1], gini_obs <= gini": bool(df.gini.between(0, 1).all() and df.gini_obs.between(0, 1).all() and (df.gini_obs <= df.gini * (1 + 1e-5)).all()),
        "hill_0 >= hill_1 >= hill_2 >= hill_inf": bool(((df.hill_0 >= df.hill_1 * tol) & (df.hill_1 >= df.hill_2 * tol) & (df.hill_2 >= df.hill_inf * tol)).all()),
        "hill_0 == n_classes": bool((df.hill_0 == df.n_classes).all()),
        "chao1 >= hill_0 (richesse observée)": bool((df.chao1 >= df.hill_0 * tol).all()),
        "hill_inf >= 1, kl_emp >= 0": bool((df.hill_inf >= tol).all() and (df.kl_emp >= 0).all()),
        "seed unique par run": bool(df.seed.is_unique),
    }
    print(f"--- {name}")
    for k, ok in checks.items():
        print(("OK   " if ok else "ÉCHEC"), k)

def plot_uniformity(df, bounds, int_bounds, title):
    fig, axes = plt.subplots(2, 3, figsize=(12, 6))
    for ax, name in zip(axes.ravel(), [*bounds, *int_bounds]):
        x = df[name].to_numpy(); lo, hi = (bounds | int_bounds)[name]
        ax.hist(x, bins=np.arange(lo - 0.5, hi + 1.5) if name in int_bounds else 50, density=True, alpha=0.7)   # entiers : un bin par valeur
        ax.axhline(1 / (hi - lo + (name in int_bounds)), color="k", ls="--", lw=1)
        if name in bounds:
            sample = np.random.default_rng(0).choice(x, size=100_000, replace=False)
            ax.set_title(f"{name} [{lo}, {hi}]  (KS p={stats.kstest((sample - lo) / (hi - lo), 'uniform').pvalue:.2f})")
        else:
            ax.set_title(f"{name} [{lo}, {hi}]  (min {x.min()}, max {x.max()})")
    fig.suptitle(title); fig.tight_layout(); plt.show()

def replay_check(df, n=30):
    sample = df.sample(n, random_state=0); errs = []
    for _, r in sample.iterrows():
        sim = Simulation(int(r.n_i), int(r.n_f), int(r.C), int(r["T"]), float(r.alpha), float(r.mu), float(r.q), int(r.seed)).run()
        assert sim.archive.sum() == r.n_f
        errs.append(max(abs(sim.metrics[k] - r[k]) / max(abs(sim.metrics[k]), 1e-12) for k in ["gini_obs", "hill_1", "hill_2", "hill_inf", "chao1", "kl_emp"]))
    exact = np.array(errs) < 1e-4
    print(f"rejeu : {exact.sum()} / {n} runs reproduits à l'identique ({'OK' if exact.mean() >= 0.9 else 'ÉCART'})")

for name, df in (("n_f = 10 000", df10), ("n_f = 30 901", df30)):
    check_properties(df, name)
    replay_check(df)
plot_uniformity(df10, {"mu": (0.0, 0.05), "q": (0.5, 1.5), "alpha": (0.0, 2.0)}, {"T": (300, 1000), "C": (2, 200), "n_i": (1, 100)}, "Uniformité des paramètres, n_f = 10 000")
plot_uniformity(df30, {"mu": (0.0, 0.02), "q": (0.5, 1.5), "alpha": (0.0, 2.0)}, {"T": (300, 1000), "C": (2, 200), "n_i": (1, 100)}, "Uniformité des paramètres, n_f = 30 901")

## 4. Grille `n_f = 10 000`

### 4.1 Où se situe l'empirique parmi les simulations ?
Quantiles de chaque indice sur les 5 M de runs, avec l'empirique et son percentile. Un percentile proche de 0 ou 100 % signalerait un indice que la grille ne couvre pas.

In [ ]:
ex10.describe(["gini", "gini_obs", "hill_0", "hill_1", "hill_2", "hill_inf", "chao1", "kl_emp"])[["5%", "50%", "95%", "empirique", "percentile_emp"]].round(3)

### 4.2 Vue globale : paramètres → indices
Lignes rouges = valeurs empiriques.

In [ ]:
ex10.pairplot(mode="cross")

### 4.3 Un indice en détail
Distribution de l'indice et dépendance (médiane, bande 10-90 %) à chaque paramètre. Exemple : Chao1 ; essayer `gini_obs`, `hill_1`, `kl_emp`.

In [ ]:
fig, summary = ex10.index("chao1")
summary.T

### 4.4 Zone de paramètres compatible avec l'empirique
Les 10 000 meilleurs runs par distance ABC, points colorés par la distance.

In [ ]:
near10 = ex10.closest(10_000, metrics=METRICS_ABC)
near10.pairplot()

### 4.5 Forme de la distribution : ajustement MLE des 100 meilleurs runs

Comparaison des exposants ajustés sur les 100 meilleurs runs de chaque sélection à celui de l'empirique. Les échantillons n'ont pas la même taille (10 000 simulés contre 30 901 empiriques) et `xmin`, `n_tail` et l'exposant en dépendent : l'empirique est donc **aminci** à 10 000 occurrences (300 tirages sans remise), et les courbes sont en fréquences relatives.

In [ ]:
N_TOP = 100
sel10 = select_top(ex10, df10, N_TOP)
counts10 = read_counts(sel10, GRIDS[10_000][0], 10_000)
fits10 = fit_all(counts10)
thin10 = thinned_empirical_fits(10_000)
lo10, hi10 = thin10.alpha.quantile([0.025, 0.975])
print(f"empirique aminci à 10 000 : alpha {qtl(thin10.alpha)} | xmin {qtl(thin10.xmin)} | n_tail {qtl(thin10.n_tail)}\n")
report_fits(fits10, lo10, hi10, f"l'IC 95 % de l'empirique aminci [{lo10:.2f} ; {hi10:.2f}]")
selection_summary(sel10, 10_000)

In [ ]:
plot_rank_frequency(counts10, fits10, 10_000, thin=thin10)

## 5. Grille `n_f = 30 901` (taille réelle des données)

L'archive finale simulée contient autant d'individus (30 901) que le corpus d'occurrences de catégories : les comparaisons de `xmin`, `n_tail`, Chao1 et de l'exposant MLE se font **sans amincir**, en comptes bruts. `mu` est rééchelonné par rapport à la grille précédente : `mu × n_f` (nombre de classes créées) est ce qui compte.

### 5.1 Où se situe l'empirique parmi les simulations ?

In [ ]:
ex30.describe(["gini", "gini_obs", "hill_0", "hill_1", "hill_2", "hill_inf", "chao1", "kl_emp"])[["5%", "50%", "95%", "empirique", "percentile_emp"]].round(3)

### 5.2 Vue globale : paramètres → indices

In [ ]:
ex30.pairplot(mode="cross")

### 5.3 Zone de paramètres compatible avec l'empirique
Les 10 000 meilleurs runs par distance ABC (`gini_obs`, Hill, Chao1), points colorés par la distance.

In [ ]:
near30 = ex30.closest(10_000, metrics=METRICS_ABC)
near30.pairplot()

### 5.4 Forme de la distribution : ajustement MLE des 100 meilleurs runs
Échantillons de même taille que l'empirique : l'intervalle de référence de l'exposant empirique est ±2σ.

In [ ]:
sel30 = select_top(ex30, df30, N_TOP)
counts30 = read_counts(sel30, GRIDS[30_901][0], 30_901)
fits30 = fit_all(counts30)
print(f"empirique : alpha {emp_fit['alpha']:.2f} ± {emp_fit['sigma']:.2f} | xmin {emp_fit['xmin']:.0f} | n_tail {emp_fit['n_tail']:.0f}\n")
report_fits(fits30, emp_fit["alpha"] - 2 * emp_fit["sigma"], emp_fit["alpha"] + 2 * emp_fit["sigma"], "±2σ de l'empirique")
selection_summary(sel30, 30_901)

In [ ]:
plot_rank_frequency(counts30, fits30, 30_901)

### 5.5 Identifiabilité des paramètres
Quantiles des paramètres parmi les 1000 meilleurs runs de chaque sélection, comparés à la plage tirée : un paramètre est contraint si sa plage dans les meilleurs runs est nettement plus étroite que la plage de départ (dernière colonne).

In [ ]:
ranges = {"q": (0.5, 1.5), "mu": (0.0, 0.02), "alpha": (0.0, 2.0), "C": (2, 200), "n_i": (1, 100), "T": (300, 1000)}
identifiability(ex30, df30, ranges)

### 5.5 bis Compromis richesse / forme
Classes à une seule occurrence des runs de KL minimal (l'empirique en a 12), et sélection en imposant aussi la richesse : `hill_0` et Chao1 à ±10 %, `gini_obs` à ±5 % de l'empirique, puis KL minimal parmi ces runs.

In [ ]:
n_single = {name: [int((c == 1).sum()) for c in cs] for name, cs in counts30.items()}
print(f"classes à 1 occurrence (empirique : {empirical_info['f1_singletons']}) : " + " | ".join(f"{k} : médiane {int(np.median(v))}" for k, v in n_single.items()))

joint = df30[(abs(df30.hill_0 / empirical["hill_0"] - 1) < 0.10) & (abs(df30.chao1 / empirical["chao1"] - 1) < 0.10)
             & (abs(df30.gini_obs / empirical["gini_obs"] - 1) < 0.05)]
print(f"runs avec hill_0 et chao1 à ±10 % et gini_obs à ±5 % : {len(joint):,} ; KL minimal parmi eux : {joint.kl_emp.min():.4f} (KL minimal global : {df30.kl_emp.min():.4f})")
sel_joint = {"richesse + forme (KL)": joint.nsmallest(N_TOP, "kl_emp")}
counts_joint = read_counts(sel_joint, GRIDS[30_901][0], 30_901)
report_fits(fit_all(counts_joint), emp_fit["alpha"] - 2 * emp_fit["sigma"], emp_fit["alpha"] + 2 * emp_fit["sigma"], "±2σ de l'empirique")
print(f"classes à 1 occurrence : médiane {int(np.median([int((c == 1).sum()) for c in counts_joint['richesse + forme (KL)']]))}")

### 5.6 Un indice en détail : `kl_emp`
Dépendance de la divergence à l'empirique à chaque paramètre.

In [ ]:
fig, summary = ex30.index("kl_emp")
summary.T

## 6. Comparaison des deux grilles

Paramètres et indices médians des 100 meilleurs runs de chaque sélection dans les deux grilles. Si `mu × n_f` est comparable, le rééchelonnement de `mu` suffit pour passer d'une taille d'archive à l'autre.

In [ ]:
comp = {}
for grid, sel, nf in (("n_f = 10 000", sel10, 10_000), ("n_f = 30 901", sel30, 30_901)):
    for name, d in sel.items():
        s = d[["q", "mu", "alpha", "C", "n_i", "T", "gini_obs", "hill_0", "hill_1", "chao1", "kl_emp"]].median()
        s["mu x n_f"] = s["mu"] * nf
        comp[(grid, name)] = s
pd.DataFrame(comp).T[["q", "mu", "mu x n_f", "alpha", "C", "n_i", "T", "gini_obs", "hill_0", "hill_1", "chao1", "kl_emp"]].round(4)

## 7. Interprétations et conclusions

*Toutes les valeurs ci-dessous sont celles des cellules précédentes (100 meilleurs runs par sélection, sauf mention contraire).*

**1. La grille couvre l'empirique.** L'empirique tombe à l'intérieur de la distribution simulée pour chaque indice (percentiles de 17 à 68 % pour `n_f = 10 000`, de 12 à 70 % pour `n_f = 30 901`) : le modèle et les bornes de tirage sont capables de reproduire les niveaux de Gini, de richesse et de Hill observés.

**2. Innovation et anti-conformisme sont nécessaires.** Chao1 dépend surtout de `mu` (Spearman +0,68) et de `q` (+0,64), et pas de `alpha`, `C`, `n_i`, `T` (≈ 0). Les meilleurs runs ont `q` ≈ 0,63–0,73 dans les deux grilles (90 % des 1000 meilleurs runs par KL entre 0,61 et 0,79 à 30 901) : **`q` < 1, c'est-à-dire un avantage aux classes rares** ; le conformisme (`q` > 1) est exclu. Et `mu` est contraint : `mu × n_f` ≈ 166–167 classes créées à 10 000, ≈ 179–211 à 30 901. Le rééchelonnement de `mu` avec `n_f` suffit donc à passer d'une taille d'archive à l'autre (à 7 à 26 % près). Attention : dans l'ancienne implémentation (v1, sans innovation), les meilleurs runs avaient `conformity_bias` ≈ 1,00 (hors notebook) ; les valeurs de `q` ne se comparent pas d'un modèle à l'autre, car ici `q` < 1 compense les classes rares créées par l'innovation.

**3. Des indices scalaires ajustés ne suffisent pas à reproduire la forme.** La sélection ABC (`gini_obs`, Hill, Chao1) donne des runs dont les indices sont très proches de l'empirique (Hill 1 ≈ 46, Chao1 ≈ 200) mais dont la queue s'écarte d'une loi de puissance : exposant MLE 1,66 (10 000) et 1,91 (30 901) contre 1,53 ± 0,05, avec seulement 12 et 7 runs sur 100 compatibles. La sélection par **KL minimal** reproduit la forme : exposant 1,56 et 1,55, `n_tail` de 101 à 108 (137 pour l'empirique), 44 et 66 runs sur 100 compatibles, et une superposition avec l'empirique sur tous les rangs (figures 4.5 et 5.4). Le KL est un bien meilleur critère de sélection que les indices résumés.

**4. Identifiabilité limitée.** `q` et `mu` sont fortement contraints (largeur 5-95 % de 13 à 19 % de la plage tirée) ; `C`, `n_i` et `T` ne le sont pas (89 à 96 % de la plage) : **les conditions initiales et la durée sont « oubliées »** par le modèle. `alpha` est partiellement contraint dans la sélection par KL (38 % de la plage) mais son 95e centile est à 1,97 pour une borne à 2,0 : la borne haute tronque probablement la distribution.

**5. Ce qui reste en défaut.** Les runs de KL minimal à 30 901 ont trop de classes rares : `hill_0` = 224 et Chao1 ≈ 256 (contre 176 et 200), `gini_obs` = 0,84 (contre 0,80). Le KL rang par rang pénalise peu ce surplus dans la queue. Forcer richesse et forme ensemble (cellule 5.5 bis) dégrade le KL et l'exposant : il y a un compromis entre la richesse en classes rares et la forme de la queue, que ce modèle (où chaque innovation est un singleton) ne résout pas.

**Limites.** (i) Les « meilleurs runs » sont un échantillon de la région compatible, pas une posterior calibrée (aucune tolérance ABC). (ii) Un seul corpus empirique, une seule réalisation (l'exposant empirique a lui-même une incertitude ±0,05). (iii) Le KL compare les rangs, pas les identités de catégories. (iv) Le rejeu local diffère du serveur sur environ 1 % des runs (versions de numpy) ; les données stockées sont cohérentes entre elles. (v) Le `gini` initial compte les classes de compte 0 (présentes dans 96 % des runs) : à ne pas utiliser pour la comparaison, d'où `gini_obs`.

**Suite possible.** (a) Relancer avec `alpha` ∈ [0 ; 4] (borne haute atteinte). (b) Ajouter à la sélection un terme sur le nombre de singletons et Chao1 en plus du KL. (c) Tester des innovations qui ne restent pas toutes des singletons, ou une part de classes créées rejoignant des classes existantes, pour lever le compromis richesse/forme. (d) Répliquer les meilleurs jeux de paramètres avec d'autres graines pour séparer bruit de simulation et effet des paramètres.

## Annexe : notes de travail d'origine

Notes conservées telles qu'elles étaient dans le notebook avant restructuration.

```R
N = 5000000
q = runif(N,.8,1.2)
r = runif(N,0,1)
T = runif(N, 300,1000)
write.cs(cbind(q,r), file='...')

```

T_1 : distributition loi de puissance
n pas : snapshot => on l'ajoute à la distribution T_1
De Cette distribution, on prend les fréquences => initialisation pour les n pas suivant 
on recommence. 

On regarde à la toute fin 

pop : État initial (dump) => 1000 histoires tagées
6 mois plus tard : 1000 histoires toujours tagées
=> Objectif  : mesurer les NOUVEAUX tags (sample aléatoire des histoires précédente ? )

Sampling random de cequi existe déjà ? 


Uniquement les nouvelles histoires qui vont changer / sur lequel le WF joue

=> couper le jeu de données de façon variable entre 50 -> 5000 périodes
Si proche de 50 :> beaucoup d'histoire d'un coup

Version abstraite avce interpolation linéaire pour avoir pop_final àa la fin
Pas de WF vraiment au final 

Population : 
-> linéaire (interpolation) -> paramétrique (en fonction des données)


Innovation: 
- Initialisation 10 (petit)
- Nouvelles histoires => associer tag < smaple aléatoire
- Proba mu de nouvelle classe => si mu=1% => Nouveau tag pas présent. sur les 50 nouvelles histoires... 
  


Pour faire le lien : 
- Question d'archivage 
- nb de génération 
- 